# Sesión 2 — Mirar antes de modelar: datos y preprocesamiento

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 1

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Clasificar cada columna de un dataset por su **tipo** y decir cómo se prepara para un modelo.
2. Encontrar los datos faltantes **visibles y disfrazados**, y preguntarse *por qué* faltan.
3. Distinguir en una gráfica un cambio **de la realidad** de un cambio **en la forma de registrar**, y decidir qué periodo usar.
4. Dejar una tabla lista para scikit-learn: categóricas a *one-hot*, numéricas escaladas, y documentar cada decisión.

> 💾 **Antes de escribir nada:** menú *Archivo → Guardar una copia en Drive* y trabaje en esa copia; si no, lo que escriba se pierde al cerrar la pestaña. Si Colab avisa que el notebook no es de Google, elija *Ejecutar de todos modos*.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
   ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: las dos primeras etapas, a fondo. El modelo solo aparece para mostrarnos lo que se nos escapó.
```

## Cómo usar este notebook
- 🔍 **Prediga**: escriba su predicción *antes* de ejecutar la celda siguiente.
- ✏️ **Complete**: reemplace los `...` (marcados con `# TODO`). Ejecute celda por celda: si usa "Ejecutar todo", el notebook se detiene en el primer ✏️ hasta que lo complete.
- 💥 **Trampa**: hoy hay una. Algo va a salir "muy bien"… y vamos a ver por qué no.
- 🖼️ **Fichas de referencia**: imágenes para consultar (y guardar) mientras trabaja en su proyecto.
- ⚡ **Mini-reto**: opcional.

> **Datos:** los mismos de la Sesión 1 — incidentes viales de Medellín 2019–2025 (Secretaría de Movilidad de Medellín; Sánchez, Arango y Correa, *Data*, 2026, CC BY 4.0). Hoy no buscamos un modelo mejor: buscamos **entender qué hay en la tabla** antes de pedirle nada.


## 0. Setup

Igual que en la Sesión 1: baja al disco de Colab el CSV y `figuras.py` (las funciones que dibujan las figuras del curso). Si ya están, no los vuelve a bajar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)

REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVO = "incidentes_medellin_2019_2025.csv"
URL_REPO = REPO + "datos/" + ARCHIVO

In [ ]:
def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo} del repositorio. Use el Plan B (celda siguiente).")
    return False

asegurar_datos(ARCHIVO, URL_REPO)
asegurar_datos("figuras.py", REPO + "tools/figuras.py")
URL = ARCHIVO
try:
    import figuras as fg
except ModuleNotFoundError:   # Plan B: GitHub no respondió y no llegó figuras.py
    from google.colab import files
    print("⚠️ No llegó figuras.py: súbalo (el profesor lo comparte por el canal del curso).")
    files.upload()
    import figuras as fg
print("Listo. pandas", pd.__version__, "| datos en:", URL)

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar el archivo</summary>

La celda siguiente revisa sola si los datos llegaron; si no, le pedirá subir `incidentes_medellin_2019_2025.csv` desde su computador (el instructor lo comparte por el grupo del curso).
</details>

In [ ]:
# Plan B: solo actúa si la descarga falló. Con "Ejecutar todo" no se detiene.
from pathlib import Path
if Path(URL).exists():
    print("✓ Datos disponibles: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ No se encontró", URL, "→ suba el archivo que compartió el instructor.")
    subido = files.upload()
    if subido:
        URL = next(iter(subido))
        print("Usando archivo local:", URL)

In [ ]:
fg.pipeline(["datos", "preprocesamiento"], modulo=1,
            subtitulo="HOY: mirar los datos antes de modelar, y dejarlos listos para un modelo");

## 1. Lo que ya sabemos hacer (la Sesión 1 en una celda)

Cargar, limpiar los nombres de columna, construir la *y*. Ya lo hizo una vez; aquí va junto para no repetirlo. Lea cada línea: ¿reconoce qué hace?

In [ ]:
import unicodedata

def limpiar_nombre(nombre):
    """'Codigo Comuna Planeacion' -> 'codigo_comuna_planeacion'; 'AÑO' -> 'ano'."""
    nombre = unicodedata.normalize("NFKD", nombre.strip().lower()).encode("ascii", "ignore").decode()
    return nombre.replace(" ", "_")

df = pd.read_excel(URL) if URL.endswith(".xlsx") else pd.read_csv(URL)
df.columns = [limpiar_nombre(c) for c in df.columns]
df = df.rename(columns={
    "llave": "id", "ano": "anio", "fecha_incidente": "fecha", "hora_incidente": "hora",
    "clase_incidente": "clase", "gravedad_incidente": "gravedad", "direccion_interpolada": "direccion",
    "codigo_comuna_planeacion": "codigo_comuna", "comuna_planeacion": "comuna",
    "codigo_barrio_planeacion": "codigo_barrio", "barrio_planeacion": "barrio",
})
df = df[df["anio"] >= 2019].reset_index(drop=True)
df["con_victimas"] = (df["gravedad"] != "SOLO DAÑOS").astype(int)    # la y de la Sesión 1
print(df.shape)
df.head(3)

En la Sesión 1 entrenamos un árbol sobre estos datos y obtuvo **0,83 de accuracy** contra **0,71** de la línea base. Parecía un buen resultado. Hoy vamos a revisarlo con lupa, pero no mirando el modelo: **mirando los datos**.

## 2. ¿Qué hay en cada columna? Los tipos de variable

Antes de cualquier modelo, cada columna se clasifica. El tipo decide cómo se prepara: una fecha no se usa igual que una comuna, y un identificador no se usa nunca.

![¿Qué tipo de variable es?](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S02/img/tipos_de_variables.png)

*Ficha de referencia: guárdela, la va a necesitar para su proyecto.*

### 👀 Mire
pandas le asigna a cada columna un **tipo de dato** (`dtype`): `int64` (enteros), `float64` (decimales) u `object` (texto). Antes de correr: ¿qué `dtype` tendrá `fecha`? ¿Y `codigo_comuna`, que parece un número?

In [ ]:
df.dtypes

### ✏️ Completa — clasifique las columnas
Complete el diccionario con el tipo de cada columna que falta, usando las categorías de la ficha: `"continua"`, `"discreta"`, `"nominal"`, `"ordinal"`, `"fecha"`, `"identificador"`, `"alta cardinalidad"`, `"objetivo"`.

In [ ]:
tipos = {
    "id": "identificador",
    "anio": "discreta",
    "fecha": "fecha",
    "hora": "fecha",
    "latitud": "continua",
    "longitud": "continua",
    "codigo_comuna": "nominal",
    "codigo_barrio": "nominal",
    "barrio": "nominal",
    "con_victimas": "objetivo",
    "clase": ...,        # TODO
    "gravedad": ...,     # TODO  (pista: ¿sus valores tienen orden?)
    "comuna": ...,       # TODO
    "direccion": ...,    # TODO  (pista: ¿cuántos valores distintos tiene? pruebe df["direccion"].nunique())
}
pd.Series(tipos).value_counts()

## 3. ¿Qué falta? Las cuatro caras del dato faltante

Un dato faltante no siempre es una celda vacía. A veces está escrito como texto (`"N/D"`), a veces es un valor imposible (`0`, `-`), y a veces falta *más en unos años que en otros*.

![Las cuatro caras del dato faltante](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S02/img/caras_del_vacio.png)

### 👀 Mire
`df.isna()` marca con `True` las celdas vacías; `.mean()` da la proporción por columna. ¿Qué columna tendrá **más** vacíos? ¿Más o menos del 10 %?

In [ ]:
(df.isna().mean() * 100).round(1).sort_values(ascending=False)

Latitud y longitud (6,1 %), barrio (5,2 %). ¿Y `comuna`? Según `isna()`, **0 %**. Pero mire sus valores:

In [ ]:
df["comuna"].value_counts().tail(5)

Ahí está: **`N/D`** (*no disponible*), 2 813 veces. Es un vacío **disfrazado** de texto: `isna()` no lo ve porque técnicamente la celda tiene algo escrito.

### ✏️ Completa — cazar disfrazados
Complete la función para que cuente, en cada columna de texto, cuántas celdas tienen alguno de los valores sospechosos.

In [ ]:
SOSPECHOSOS = ["N/D", "ND", "SIN INFORMACION", "NO APLICA", "0", "-", ""]

def contar_disfrazados(df, sospechosos=SOSPECHOSOS):
    conteo = {}
    for col in df.select_dtypes(include=["object", "string"]).columns:
        valores = df[col].astype(str).str.strip().str.upper()
        n = ...   # TODO: ¿cuántos de `valores` están en la lista `sospechosos`? (pista: .isin(...).sum())
        if n > 0:
            conteo[col] = n
    return pd.Series(conteo, name="disfrazados").sort_values(ascending=False)

contar_disfrazados(df)

Ahora todo junto en una imagen. A la izquierda cada fila es un incidente, ordenados en el tiempo; gris oscuro = vacío real, rojo = disfrazado.

In [ ]:
columnas_originales = ["id", "anio", "fecha", "hora", "clase", "gravedad", "direccion", "latitud",
                       "longitud", "codigo_comuna", "comuna", "codigo_barrio", "barrio"]
fg.mapa_vacios(df[columnas_originales]);

### 🔍 Predice — ¿el vacío informa?
En todo el dataset, el **71 %** de los incidentes tiene víctimas. En las filas donde `comuna = "N/D"`, ¿qué porcentaje cree que tiene víctimas?

- (a) más o menos lo mismo, ~71 % · (b) bastante menos · (c) bastante más

**Su predicción:** ____

In [ ]:
grupos = {
    "comuna = N/D": df["comuna"] == "N/D",
    "sin latitud": df["latitud"].isna(),
    "sin barrio": df["barrio"].isna(),
}
for nombre, mascara in grupos.items():
    print(f"{nombre:<14} {mascara.sum():>6} filas   {df.loc[mascara, 'con_victimas'].mean():.0%} con víctimas")
print(f"{'todos':<14} {len(df):>6} filas   {df['con_victimas'].mean():.0%} con víctimas")

**92 %** con víctimas cuando la comuna es `N/D`, contra 71 % en general. Si dejáramos `N/D` como una comuna más, el árbol aprendería *"comuna desconocida ⇒ víctimas"*. ¿Es eso un patrón de la ciudad? Antes de creerlo, una pregunta más: **¿de cuándo son esas filas?**

In [ ]:
fg.vacio_informa(df, "con_victimas", grupos, grupo_anio="comuna = N/D");

## 4. ¿Cómo se reparte? Mirar las distribuciones

Primero, convertir fecha y hora en números (como en la Sesión 1).

In [ ]:
df["fecha"] = pd.to_datetime(df["fecha"])
df["hora_num"] = df["hora"].astype(str).str.slice(0, 2).astype(int)   # "07:35" -> 7
df["dia_semana"] = df["fecha"].dt.dayofweek                          # 0 = lunes … 6 = domingo
df["mes"] = df["fecha"].dt.month
df[["hora_num", "dia_semana", "mes", "anio"]].describe().round(1)

### 🔍 Predice
1. ¿A qué hora del día cree que ocurren **más** incidentes?
2. ¿Y a qué hora son **más graves** (mayor % con víctimas)? ¿Es la misma hora?

**Su predicción:** más incidentes a las ____ · más graves a las ____

In [ ]:
fg.distribucion_hora(df);

### 👀 Mire
Entre enero de 2019 y agosto de 2025, ¿qué **mes** tuvo **menos** incidentes registrados? Piense un candidato (y por qué) antes de correr.

In [ ]:
por_mes = df.set_index("fecha").resample("MS").size()
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "octubre", "noviembre", "diciembre"]
nombre = lambda f: f"{MESES[f.month - 1]} de {f.year}"
print("Mes con menos incidentes:", nombre(por_mes.idxmin()), "→", por_mes.min())
print("Mes con más incidentes:  ", nombre(por_mes.idxmax()), "→", por_mes.max())
print("Mediana mensual:", int(por_mes.median()))

## 5. 💥 El árbol que aprendió el calendario

Volvamos al modelo de la Sesión 1. Lo entrenamos otra vez, idéntico, y miramos qué preguntas hace.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

def entrenar_arbol_S1(datos):
    """El pipeline de la Sesión 1, empaquetado: devuelve (accuracy línea base, accuracy árbol, árbol, X)."""
    columnas_x = ["hora_num", "dia_semana", "mes", "anio", "comuna", "clase"]
    d = datos[columnas_x + ["con_victimas"]].dropna()
    X = pd.get_dummies(d[columnas_x], columns=["comuna", "clase"], dtype=int)
    y = d["con_victimas"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    arbol = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_tr)
    tonto = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
    return tonto.score(X_te, y_te), arbol.score(X_te, y_te), arbol, X

base_todo, acc_todo, arbol_todo, X_todo = entrenar_arbol_S1(df)
print(f"Línea base: {base_todo:.3f}   Árbol: {acc_todo:.3f}   → le gana por {100*(acc_todo-base_todo):.1f} puntos")

In [ ]:
fg.arbol_reglas(arbol_todo, X_todo.columns, max_depth=2);

0,83 contra 0,71. ¡Le gana por 12 puntos! Y la segunda pregunta que hace el árbol es… **`anio <= 2022.5`**.

### 🔍 Predice
El árbol encontró que "ser de 2023 en adelante" separa muy bien con víctimas de solo daños. Antes de ver la gráfica, elija:

- (a) Desde 2023 Medellín tiene incidentes mucho más graves (más motos, más velocidad…).
- (b) Desde 2023 algo cambió en **cómo se registran** los incidentes.
- (c) Es casualidad del árbol.

**Su predicción:** ____ · ¿Qué esperaría ver en una gráfica de incidentes por mes si fuera (a)? ¿Y si fuera (b)?

### 🖼️ Las tres huellas
Antes de mirar la serie real, dibujemos cómo se vería la gráfica de incidentes por mes **bajo cada hipótesis** (datos inventados: lo que importa es la forma, no los números). Mire el rojo después del corte.

In [ ]:
fg.tres_hipotesis();

Con las tres huellas en la cabeza, ¿mantiene su predicción? Ahora sí, la serie real:

In [ ]:
fg.serie_mensual(df, marcas=[("2020-03-25", "cuarentena"), ("2022-10-01", "octubre 2022")]);

### Segunda vuelta: ¿y el `N/D` de la sección 3?
Las filas con `comuna = "N/D"` tenían 92 % de víctimas, y casi todas eran de 2025. Ya sabemos que en 2025 casi todo lo registrado tiene víctimas. ¿Qué pasa con el `N/D` si miramos solo el periodo **antes** del cambio?

In [ ]:
antes = df[df["fecha"] < "2022-10-01"]
nd = antes["comuna"] == "N/D"
print(f"Antes de oct-2022:  {nd.sum()} filas con N/D → {antes.loc[nd, 'con_victimas'].mean():.0%} con víctimas "
      f"(vs {antes['con_victimas'].mean():.0%} en general)")

Del 92 % al **51 %**, que es menos que el promedio del periodo (58 %). El "patrón" del `N/D` también era el calendario. Dos hallazgos espectaculares, **cero** sobre accidentes.

### La decisión: con qué periodo trabajamos

Si la pregunta es *"¿qué hace que un incidente tenga víctimas?"*, necesitamos un periodo donde **los dos tipos de incidente se registraban**. Desde octubre de 2022 eso ya no pasa. Decisión del curso para la pregunta de gravedad:

> **`df_grav` = incidentes de enero de 2019 a septiembre de 2022.**

Los datos de después no se botan: sirven para otras preguntas (¿cuántos heridos por mes?, ¿dónde?), pero no para esta.

### 🔍 Predice
Entrenamos **el mismo árbol** sobre `df_grav`. Comparado con el 0,83 de antes:

1. ¿El accuracy del árbol **sube o baja**?
2. ¿Y la distancia entre el árbol y la línea base (los "puntos que le gana")?

**Su predicción:** accuracy ____ · ventaja sobre la línea base ____

In [ ]:
df_grav = df[df["fecha"] < "2022-10-01"].copy()
print(df_grav.shape, f"→ {df_grav['con_victimas'].mean():.1%} con víctimas")

base_grav, acc_grav, arbol_grav, X_grav = entrenar_arbol_S1(df_grav)
fg.vs_linea_base({"2019–2025\n(todo)": (base_todo, acc_todo),
                  "ene 2019 – sep 2022\n(df_grav)": (base_grav, acc_grav)});

### ¿Cómo comparar dos modelos con líneas base distintas?
Los "puntos que le gana" (+12 antes, +19 ahora) **no** se pueden comparar directamente: la línea base cambió (0,71 → 0,58) y con una línea base más baja hay más espacio para ganar. Lo comparable es **qué parte del error del tonto elimina el modelo**:

$$rac{	ext{accuracy} - 	ext{línea base}}{1 - 	ext{línea base}}$$

Con todos los datos: (0,831 − 0,712) / 0,288 ≈ **41 %**. Con `df_grav`: (0,769 − 0,578) / 0,422 ≈ **45 %**. Parecidos: el árbol no "aprendió mucho más"; lo que cambió es que ahora su acierto viene de los incidentes y no del calendario. Un accuracy solo no dice nada; dice algo comparado con la línea base y sobre los datos correctos.

### Lección de la trampa
1. Un resultado muy bueno es una razón para **sospechar**, no para celebrar.
2. Antes de modelar, grafique la *y* en el tiempo. Un salto brusco suele ser un cambio de **registro**, no de la realidad.
3. Las columnas de fecha (`anio`, `mes`) pueden "predecir" por razones que no tienen nada que ver con el fenómeno. Cada vez que el modelo diga que el tiempo importa, pregunte: ¿cambió el mundo o cambió el dato?

## 6. Preprocesamiento: dejar la tabla lista para un modelo

Ya sabemos *qué* hay y *qué periodo* usar. Ahora, las tres operaciones que va a repetir en todos sus proyectos. Todo sobre `df_grav`.

### 6.1 ¿Qué hago con los vacíos?

| Opción | Qué hace | Cuándo sirve | Riesgo |
|---|---|---|---|
| **Quitar filas** (`dropna`) | borra las filas con vacío | pocos vacíos y sin patrón | si el vacío tiene patrón, sesga los datos |
| **Imputar** (`fillna`) | rellena con la mediana / la moda | pocos vacíos en numéricas | inventa valores "típicos" |
| **Imputar + marcar** | rellena *y* crea una columna `falta_x` (0/1) | cuando el vacío puede informar | el modelo puede aprender el patrón del vacío (¡verifique por qué falta!) |
| **Recodificar** | convierte `"N/D"` en `NaN` o en una categoría `"desconocida"` | vacíos disfrazados | — |

Primero, los disfrazados se convierten en vacíos de verdad. Luego, para las coordenadas, imputamos con la mediana **y** marcamos. Así se ve la operación sobre seis filas:

In [ ]:
fg.imputar_y_marcar();

Fíjese en la columna `falta_coord`: es la que conserva la información de *que faltaba*. Sin ella, el modelo no distingue una fila imputada de una fila medida.

In [ ]:
df_grav = df_grav.replace({"comuna": {"N/D": np.nan}, "codigo_comuna": {"N/D": np.nan},
                           "barrio": {"0": np.nan}, "codigo_barrio": {"-": np.nan, "0": np.nan}})
print("vacíos en comuna ahora:", df_grav["comuna"].isna().sum())

### ✏️ Completa — imputar y marcar
Cree la columna `falta_coord` (1 si `latitud` está vacía) y luego rellene `latitud` y `longitud` con su **mediana**.

In [ ]:
df_grav["falta_coord"] = ...   # TODO: 1 si latitud es NaN, 0 si no (pista: .isna().astype(int))
for col in ["latitud", "longitud"]:
    df_grav[col] = ...       # TODO: rellenar con la mediana (pista: .fillna(df_grav[col].median()))
print("falta_coord:", df_grav["falta_coord"].sum(), "filas | vacíos que quedan en latitud:", df_grav["latitud"].isna().sum())

Una advertencia: aquí calculamos la mediana con **todos** los datos, incluidas las filas que después serán de prueba. Es una **fuga pequeña** (la mediana casi no cambia), pero es una fuga: en la Sesión 3 la corregimos calculando todo lo que se aprende solo con entrenamiento. Quédese con la pregunta de la ficha: *¿qué se conoce antes y qué se conoce después?*

### 6.2 Categóricas: *one-hot encoding*

Un modelo no entiende `"Choque"` ni `"El Poblado"`. ¿Por qué no numerarlas (Choque = 1, Atropello = 2, …)? Porque el modelo creería que Atropello es "el doble" de Choque, o que está "entre" Choque y Caída. La solución estándar para categóricas **sin orden** es *one-hot*: una columna de 0/1 por categoría.

In [ ]:
fg.one_hot_ejemplo(df_grav, "clase");

En la celda siguiente usamos `dummy_na=True`: además de una columna por categoría, `get_dummies` **agrega una columna `_nan` por variable** (`comuna_nan`, `clase_nan`) que vale 1 donde el valor está vacío. Así el vacío de comuna se vuelve su propia categoría en vez de borrar la fila (la opción "recodificar" de la tabla).

### 👀 Mire
`clase` tiene 6 categorías y `comuna` (ahora sin `N/D`) tiene 21. Con 4 columnas numéricas (`hora_num`, `dia_semana`, `mes`, `falta_coord`) y las dos `_nan`, ¿cuántas columnas tendrá X? Haga la cuenta y compruébela en la salida.

In [ ]:
columnas_x = ["hora_num", "dia_semana", "mes", "falta_coord", "comuna", "clase"]
X = pd.get_dummies(df_grav[columnas_x], columns=["comuna", "clase"], dtype=int, dummy_na=True)
y = df_grav["con_victimas"]
print("X:", X.shape)
[c for c in X.columns if c.startswith("comuna_")][-3:]

Para la variable **ordinal** (`gravedad`) el *one-hot* perdería el orden. Ahí se usan números que lo respeten (no la usamos como X porque de ella sale la *y*, pero así se haría):

In [ ]:
orden_gravedad = {"SOLO DAÑOS": 0, "HERIDO": 1, "MUERTO": 2}
df_grav["gravedad_num"] = df_grav["gravedad"].map(orden_gravedad)
df_grav.groupby("gravedad")["gravedad_num"].first()

### 6.3 Numéricas: escalar

Muchos modelos (los que miden **distancias**, como k-NN en la Sesión 5 y K-means en la Sesión 7, y los que **penalizan coeficientes**, como la regresión regularizada en la Sesión 4) se confunden cuando las variables varían en **rangos** muy distintos: `hora_num` va de 0 a 23, mientras `latitud` solo se mueve en centésimas (de 6,19 a 6,31). En una distancia sin escalar manda la de rango más amplio, no la de números más grandes. Escalar con `StandardScaler` pone todas en la misma unidad:

$$z = \frac{x - \text{media}}{\text{desviación estándar}}$$

Un $z = 1{,}5$ se lee: *"este valor está 1,5 desviaciones por encima del promedio"*.

### 📐 Fundamento: qué aprende y qué aplica el escalador
| | |
|---|---|
| **Qué aprende** (`.fit`) | dos números por columna: la media $\bar{x}$ y la desviación $s$ de `hora_num` |
| **Qué aplica** (`.transform`) | a cada valor: $z = (x - \bar{x}) / s$ |
| **Qué garantiza** | la columna escalada queda con media 0 y desviación 1, y **conserva la forma**: el orden y las distancias relativas entre valores no cambian |

<details><summary><b>Para quien quiera más:</b> por qué la media queda en 0 y la desviación en 1</summary>

Si a todos los valores les resto la media, el promedio de las diferencias es $\frac{1}{n}\sum (x_i - \bar{x}) = \bar{x} - \bar{x} = 0$. Si además divido por $s$, la desviación de los $z$ es $s / s = 1$. Y como resté y dividí por *constantes*, un valor que estaba por encima de otro sigue estándolo: por eso un árbol (que solo compara *mayor / menor que un umbral*) no nota la diferencia, y un modelo de distancias (k-NN, K-means) sí, porque la distancia entre dos puntos cambia según la unidad.
</details>

### 👀 Mire
Después de escalar `hora_num` (media ≈ 13,1 h, desviación ≈ 5,4 h): ¿cuál será la **media** y la **desviación** de la columna escalada? ¿Qué $z$ tendrá la hora 23, más o menos?

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler()
z = escalador.fit_transform(df_grav[["hora_num"]])
print(f"media aprendida: {escalador.mean_[0]:.2f}   desviación aprendida: {escalador.scale_[0]:.2f}")
print(f"después de escalar → media {z.mean():.2f}, desviación {z.std():.2f}, máximo {z.max():.2f}, mínimo {z.min():.2f}")

In [ ]:
fg.escalado_antes_despues(df_grav, ["anio", "hora_num", "latitud", "longitud"]);

**¿Y el árbol?** Al árbol **no le importa** la escala: pregunta "¿hora_num ≤ 5,5?" o "¿z ≤ −1,4?" y obtiene exactamente los mismos cortes. Por eso en la Sesión 1 no escalamos. Escalar no es un paso obligatorio del pipeline: es un paso que depende del modelo. La ficha de cada modelo del curso le va a decir si lo necesita.

## 7. Su herramienta para el proyecto: `explorar(df)`

Todo lo de hoy, en una función que va a correr sobre **su** dataset para la entrega E1 (se trabaja en la Sesión 3 y se entrega antes de la Sesión 4).

![Lista de chequeo antes de modelar](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S02/img/checklist_antes_de_modelar.png)

<details><summary>La lista de chequeo en texto (para copiarla en su notebook de E1)</summary>

1. **¿Qué es una fila?** Un incidente, una persona, un municipio, un mes… Si no lo puede decir en una frase, no siga.
2. **¿Cuál es la *y* y qué la define?** La regla exacta y su proporción (clases) o su rango (número).
3. **¿De qué tipo es cada columna?** Numérica, categórica, fecha, identificador, alta cardinalidad.
4. **¿Qué falta, y por qué?** NaN, disfrazados (`"N/D"`, `"SIN INFORMACION"`), imposibles (0, −1, 9999) y si se concentran en algún grupo o año.
5. **¿Cambió algo en el tiempo?** Grafique la *y* por mes: un salto brusco suele ser un cambio de registro.
6. **¿Qué columnas se conocen DESPUÉS de la *y*?** Esas no pueden ser X (Sesión 3).
7. **¿Qué decidí y por qué?** Filas, años o columnas descartadas y vacíos imputados, una línea cada uno.
</details>

### ✏️ Completa
Faltan dos columnas del resumen: el número de valores distintos y el % de vacíos reales.

In [ ]:
def explorar(df, y=None, sospechosos=SOSPECHOSOS):
    """Resumen de una tabla antes de modelar: una fila por columna."""
    print(f"Filas: {len(df):,}   Columnas: {df.shape[1]}".replace(",", " "))
    resumen = pd.DataFrame(index=df.columns)
    resumen["dtype"] = df.dtypes.astype(str)
    resumen["n_distintos"] = ...   # TODO: valores distintos por columna (pista: df.nunique())
    resumen["%_vacios"] = ...      # TODO: % de NaN por columna, redondeado a 1 decimal
    disfrazados = contar_disfrazados(df, sospechosos)
    resumen["%_disfrazados"] = (disfrazados.reindex(df.columns).fillna(0) / len(df) * 100).round(1)
    resumen["ejemplo"] = df.iloc[0].astype(str).str.slice(0, 25)
    resumen["alerta"] = ""
    resumen.loc[resumen["n_distintos"] == len(df), "alerta"] = "¿identificador?"
    resumen.loc[resumen["dtype"].isin(["object", "str", "string"]) & (resumen["n_distintos"] > 0.1 * len(df)), "alerta"] = "¿alta cardinalidad?"
    resumen.loc[resumen["%_vacios"] + resumen["%_disfrazados"] > 5, "alerta"] += " ¿por qué falta?"
    if y is not None:
        print(f"\nVariable objetivo «{y}»:")
        print(df[y].value_counts(normalize=True).round(3) if df[y].nunique() <= 10 else df[y].describe().round(2))
    return resumen

explorar(df[columnas_originales + ["con_victimas"]], y="con_victimas")

Mire la columna `alerta`: `id` sale como *¿identificador?*, `direccion` como *¿alta cardinalidad?* y las coordenadas como *¿por qué falta?*. La función no decide por usted; le dice **dónde mirar**.

### 📦 Entrega E1 (se entrega antes de la Sesión 4, por el canal que indique el profesor) — empiece hoy
1. Elija el dataset de su proyecto (lista curada en el Notebook 0). Mínimo 1 000 filas.
2. Cárguelo en Colab (desde su Drive, ver README del repositorio) y corra `explorar(su_df, y="su_variable")`.
3. Responda en el notebook las 7 preguntas de la lista de chequeo, una línea cada una.
4. Grafique su *y* en el tiempo si tiene fechas. ¿Hay algún salto como el de octubre de 2022?

En la Sesión 3 se agrega la última pieza de E1: la lista de columnas que se conocen **después** de la *y*.

## ⚡ Mini-reto (opcional)

¿El cambio de registro de octubre de 2022 pasó **en todas las comunas** o solo en algunas? Si pasó solo en algunas, podría ser un cambio de la realidad local; si pasó en todas a la vez, es casi seguro un cambio del sistema de registro.

Calcule, para cada comuna, el % de *solo daños* entre enero y septiembre de 2022 y entre octubre y diciembre de 2022. ¿Hay alguna comuna donde no cayera?

<details><summary>Pista</summary>

```python
d = df[(df["fecha"] >= "2022-01-01") & (df["fecha"] < "2023-01-01")].copy()
d["despues"] = d["fecha"] >= "2022-10-01"
d["solo_danos"] = 1 - d["con_victimas"]
d.groupby(["comuna", "despues"])["solo_danos"].mean().unstack().round(2)
```
</details>

In [ ]:
# Su código aquí

## 8. Cierre — qué cambió hoy en el pipeline

| Etapa | Sesión 1 | Hoy (Sesión 2) |
|---|---|---|
| Datos | `read_csv`, `shape`, `head` | tipos de variable · vacíos reales y disfrazados · distribuciones · **la y en el tiempo** · elegir el periodo (`df_grav`) |
| Preprocesamiento | `dropna`, `get_dummies` | recodificar disfrazados · imputar + marcar · one-hot con `dummy_na` · ordinales · `StandardScaler` |
| Modelo | árbol | (el mismo árbol, como detector de problemas) |
| Evaluación | accuracy vs línea base | comparar con la línea base: **qué parte del error del tonto elimina** el modelo |
| Interpretación | importancia de variables | si el modelo dice que la fecha importa: ¿cambió el mundo o el dato? |

**La frase de hoy:** *menos accuracy, modelo más honesto.* El de la Sesión 1 sacaba 0,83 apoyándose en el calendario; el de hoy saca 0,77 apoyándose en los incidentes. Medido como parte del error del tonto que elimina, los dos están cerca (41 % y 45 %): lo que cambió es *de dónde* sale el acierto.

**Para la Sesión 3 — entrenar, probar y no hacerse trampa:**
- Traiga elegido el dataset de su proyecto y corra `explorar()` sobre él.
- Pregunta para pensar: el escalador y la mediana aprendieron números de **todos** los datos. ¿Eso es un problema? ¿Cuándo?
